# EWC Space Delta-V · Issue 028 · Working Demo
**Three small experiments you can run in the browser (Runtime → Run all; about a minute in total).**

1. **GPS-free positioning from radio beacons**: how UWB ranging to beacons turns distances into a position, and why beacon geometry (DOP) matters.
2. **Why a night-vision satellite looks in the thermal infrared**: Planck radiance of a ~300 K body in the LWIR band versus the visible band.
3. **Why a laser link from orbit is hard at the ground end**: how atmospheric turbulence limits how much light a ground telescope can put into a single-mode fibre, and why mode converters are interesting.

These are teaching models built from first principles. They are not the VaMEx-ARES or Anduril implementations, whose internals are not public.

## Part 1 · Trilateration from beacon ranges
A rover at unknown position **x** measures ranges $r_i = \lVert \mathbf{x}-\mathbf{b}_i\rVert + n_i$ to beacons at known positions $\mathbf{b}_i$, with noise $n_i$. Gauss-Newton linearises around a guess and solves a least-squares step. The position error scales as $\sigma_x \approx \mathrm{DOP}\,\sigma_r$, where $\mathrm{DOP}=\sqrt{\mathrm{tr}\,(H^TH)^{-1}}$ and $H$ holds the unit vectors from beacons to the rover.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(28)

In [ ]:
def ranges(x, beacons, sigma, rng):
    return np.linalg.norm(beacons - x, axis=1) + rng.normal(0, sigma, len(beacons))

def trilaterate(r, beacons, x0, iters=20):
    x = x0.astype(float).copy()
    for _ in range(iters):
        d = np.linalg.norm(beacons - x, axis=1)
        H = (x - beacons) / d[:, None]
        dx, *_ = np.linalg.lstsq(H, r - d, rcond=None)
        x += dx
        if np.linalg.norm(dx) < 1e-9:
            break
    return x

def dop(x, beacons):
    d = np.maximum(np.linalg.norm(beacons - x, axis=1), 1e-6)
    H = (x - beacons) / d[:, None]
    return float(np.sqrt(np.trace(np.linalg.inv(H.T @ H))))

In [ ]:
sigma = 0.10   # ranging noise, metres (assumed UWB-class)
x_true = np.array([40.0, 30.0])
good = np.array([[0,0],[100,0],[100,100],[0,100]], float)   # beacons surround the rover
poor = np.array([[0,0],[10,0],[20,0],[30,0]], float)        # beacons bunched on one side

def monte_carlo(beacons, n=2000):
    err = []
    for _ in range(n):
        r = ranges(x_true, beacons, sigma, rng)
        est = trilaterate(r, beacons, x0=np.array([50.0, 50.0]))
        err.append(np.linalg.norm(est - x_true))
    return np.array(err)

for name, b in [("surrounding", good), ("bunched", poor)]:
    e = monte_carlo(b)
    print(f"{name:12s} DOP={dop(x_true,b):5.2f}  RMS error={np.sqrt(np.mean(e**2)):.3f} m  (sigma*DOP={sigma*dop(x_true,b):.3f} m)")

In [ ]:
xs = np.linspace(-250, 350, 181); ys = np.linspace(-250, 350, 181)
G = np.array([[min(dop(np.array([x,y]), good),6) for x in xs] for y in ys])
fig, ax = plt.subplots(figsize=(6.8, 5.6), dpi=96)
im = ax.contourf(xs, ys, G, levels=np.linspace(1, 6, 21), cmap="viridis_r")
ax.scatter(*good.T, c="k", marker="^", s=70, label="beacons")
ax.scatter(*x_true, c="w", edgecolor="k", s=60, label="rover")
fig.colorbar(im, label="2-D DOP (capped at 6)")
ax.set_xlabel("x (m)"); ax.set_ylabel("y (m)"); ax.set_aspect("equal"); ax.legend(loc="lower right")
ax.set_title("Position error multiplier across the map")
plt.show()

**Reading the result.** With four beacons surrounding the rover the DOP is about 1.0, so the position error is about the ranging error; with the same four beacons bunched in a line it is about 2.3, and across the map it stays close to 1 in and just around the square, then climbs (about 2.5 at three beacon-spacings out) as the beacons crowd into one direction. Add the 10-minute delay of a Mars link and the rover cannot ask Earth for help, so the geometry has to be right on site.

## Part 2 · Visible versus thermal infrared
Planck's law gives spectral radiance $B_\lambda(T)=\dfrac{2hc^2}{\lambda^5}\,\dfrac{1}{e^{hc/\lambda k_BT}-1}$. Wien's law puts the peak at $\lambda_{max}=b/T$, with $b=2897.77\ \mu$m K. A ~300 K body peaks near 9.7 µm, inside the 8–14 µm LWIR window, so it emits there with or without sunlight.

In [ ]:
h = 6.62607015e-34; c = 2.99792458e8; kB = 1.380649e-23; b_wien = 2.897771955e-3

def planck(lam, T):
    return 2*h*c**2 / lam**5 / np.expm1(h*c/(lam*kB*T))

_trapz = getattr(np, 'trapezoid', None) or np.trapz   # works on NumPy 1.x and 2.x

def band_radiance(l1, l2, T, n=4000):
    lam = np.linspace(l1, l2, n)
    return _trapz(planck(lam, T), lam)   # W m^-2 sr^-1

T = 300.0
print(f"Wien peak at {T:.0f} K: {b_wien/T*1e6:.2f} um")
lwir = band_radiance(8e-6, 14e-6, T)
vis  = band_radiance(0.40e-6, 0.70e-6, T)
print(f"LWIR 8-14 um radiance : {lwir:.3e} W/m^2/sr")
print(f"Visible 0.4-0.7 um    : {vis:.3e} W/m^2/sr")
print(f"Ratio LWIR/visible    : {lwir/vis:.3e}")

In [ ]:
lam = np.logspace(np.log10(0.3e-6), np.log10(30e-6), 600)
fig, ax = plt.subplots(figsize=(6.8, 4.6), dpi=96)
for T_, col in [(300,"C0"),(1000,"C1"),(5772,"C3")]:
    ax.loglog(lam*1e6, planck(lam, T_), col, label=f"{T_} K")
ax.axvspan(8, 14, color="grey", alpha=.25, label="LWIR 8-14 um")
ax.axvspan(0.4, 0.7, color="gold", alpha=.25, label="visible")
ax.set_ylim(1e-2, 1e14)
ax.set_xlabel("Wavelength (um)"); ax.set_ylabel("Spectral radiance (W m$^{-2}$ sr$^{-1}$ m$^{-1}$)")
ax.legend(); ax.set_title("Planck curves: a 300 K body shines in LWIR, not in visible")
plt.show()

**Reading the result.** At 300 K the visible-band emission is effectively zero (tens of orders of magnitude below LWIR), so a visible camera in eclipse sees nothing of a warm object that is not sunlit, while an LWIR sensor still sees its emission. Spacecraft emissivity, optics and detector noise are not modelled here.

## Part 3 · Turbulence and single-mode fibre coupling
A light wave that crosses the atmosphere arrives with a distorted wavefront. The strength of the distortion is summarised by the **Fried parameter** $r_0$ (the aperture size over which the wavefront is roughly flat). Two order-of-magnitude rules follow for a telescope of diameter $D$ when $D \gg r_0$:

$$N_{modes}\approx (D/r_0)^2 \qquad\qquad \eta_{SMF}\approx (r_0/D)^2$$

$N_{modes}$ is the number of spatial modes the light populates, and $\eta_{SMF}$ is the fraction of power a single-mode fibre accepts if nothing corrects the wavefront. Here we **simulate** Kolmogorov turbulence and measure $\eta_{SMF}$ directly, to check the rule. This is a teaching model: no adaptive optics, a flat top-hat aperture, a Gaussian fibre mode with a fixed waist of $0.64$ of the aperture radius, and no scintillation (amplitude fluctuations).

### Step 1 · Functions: a Kolmogorov phase screen and the fibre-coupling efficiency
The screen has a high-frequency part from an FFT and a low-frequency part from sub-harmonics (Lane et al., 1992), which restore the large, slow distortions an FFT grid cannot hold. Lengths are in units of the aperture diameter $D$.

In [ ]:
N = 512; DPIX = 32; DX = 1.0 / DPIX; A = DPIX / 2       # grid size, pixels across the aperture, pixel size (units of D), aperture radius (pixels)
M = DPIX // 2 + 2                                         # half-width of the cropped region around the aperture

def phase_screen(r0, rng, n_sub=6):
    f = np.fft.fftfreq(N, DX); fx, fy = np.meshgrid(f, f); fr = np.hypot(fx, fy); fr[0, 0] = 1.0
    psd = 0.023 * r0**(-5/3) * fr**(-11/3); psd[0, 0] = 0.0   # Kolmogorov phase power spectrum
    df = 1.0 / (N * DX)
    cn = (rng.standard_normal((N, N)) + 1j * rng.standard_normal((N, N))) * np.sqrt(psd) * df
    hi = np.real(np.fft.ifft2(cn)) * N * N
    c = N // 2; hi = hi[c-M:c+M, c-M:c+M]
    L = N * DX; xs = np.arange(-M, M) * DX; X, Y = np.meshgrid(xs, xs); lo = np.zeros_like(hi)
    for p in range(1, n_sub + 1):
        dfp = 1.0 / (3**p * L)
        for i in (-1, 0, 1):
            for j in (-1, 0, 1):
                if i == 0 and j == 0: continue
                fxp, fyp = i*dfp, j*dfp; fp = np.hypot(fxp, fyp)
                amp = np.sqrt(0.023 * r0**(-5/3) * fp**(-11/3)) * dfp
                cc = (rng.standard_normal() + 1j * rng.standard_normal()) * amp
                lo += np.real(cc * np.exp(2j*np.pi*(fxp*X + fyp*Y)))
    return hi + lo

yy, xx = np.indices((2*M, 2*M)); rr2 = (xx - M + 0.5)**2 + (yy - M + 0.5)**2
MASK = (rr2 <= A**2).astype(float)
FIBRE_MODE = np.exp(-(rr2 / A**2) / 0.64**2) * MASK        # Gaussian fundamental mode, waist 0.64 x aperture radius

def coupling_efficiency(screen):
    field = MASK * np.exp(1j * screen)
    return abs(np.sum(field * FIBRE_MODE))**2 / (MASK.sum() * np.sum(FIBRE_MODE**2))

### Step 2 · Check the screens, then run the experiment
First a sanity check: the phase structure function at a separation of one $r_0$ should be close to $6.88$ for Kolmogorov turbulence. Then we average the coupling efficiency over many random screens for several values of $D/r_0$.

In [ ]:
rng = np.random.default_rng(28)
sh = int(round(1.0 / DX))
d = []
for _ in range(150):
    s = phase_screen(1.0, rng); d.append(np.mean((s[:, sh:] - s[:, :-sh])**2))
print(f"Structure function at one r0: {np.mean(d):.2f}  (theory 6.88; low because the grid is finite and the sample is small)")

D_over_r0 = [0.5, 1, 2, 3, 5, 8]
eta_sim = []
for q in D_over_r0:
    vals = [coupling_efficiency(phase_screen(1.0 / q, rng)) for _ in range(150)]
    eta_sim.append(np.mean(vals))
    print(f"D/r0 = {q:>3}: simulated eta = {eta_sim[-1]:.4f}   rule (r0/D)^2 = {q**-2:.4f}")

In [ ]:
q = np.array(D_over_r0, float)
fig, ax = plt.subplots(figsize=(6.8, 4.4), dpi=96)
ax.loglog(q, eta_sim, "o-", label="simulated (no correction)")
ax.loglog(q, np.minimum(q**-2.0, 1), "--", label="rule $(r_0/D)^2$, capped at 1")
ax.set_xlabel("D / r0"); ax.set_ylabel("Single-mode fibre coupling efficiency"); ax.legend(); ax.grid(True, which="both", alpha=.3)
ax.set_title("Turbulence loss at the ground end of a laser link")
plt.show()

**Reading the result.** For $D/r_0$ of about 3 and above, the simulated efficiency follows $(r_0/D)^2$ closely; below that the rule overshoots, because a nearly flat wavefront couples well. A mode converter, such as the multi-plane light conversion devices described in the arXiv tutorial by Zhang and Fontaine, is interesting because it can take the many populated modes and route them to several single-mode outputs instead of discarding most of the light, at the price of its own insertion loss.

### Step 3 · A worked example with assumed values
$r_0$ depends on wavelength as $\lambda^{6/5}$. Starting from an assumed 5 cm at 500 nm and a 1 m telescope at 1,550 nm (assumed values, not those of any real ground station):

In [ ]:
r0_500 = 0.05                       # assumed Fried parameter at 500 nm, metres
lam = 1550e-9; D = 1.0              # assumed link wavelength and telescope diameter
r0 = r0_500 * (lam / 500e-9)**(6/5)
n_modes = (D / r0)**2
eta = (r0 / D)**2
print(f"r0 at 1550 nm = {r0:.3f} m")
print(f"Populated modes ~ {n_modes:.0f}")
print(f"Single-mode coupling ~ {eta:.3f} = {10*np.log10(eta):.1f} dB (no correction)")